# Three-Player Game

This notebook demonstrates a three-player simultaneous game with multiple reward matrix configurations.

## Game Configuration

The `ThreePlayers` game supports multiple reward matrix configurations through the `config` parameter:
- `config=1`: First reward matrix configuration
- `config=2`: Second reward matrix configuration (default)
- etc.

You can easily add more configurations by extending the `REWARD_MATRICES` dictionary in the source file.

In [1]:
from games.threeplayers import ThreePlayers
from agents.regretmatching import RegretMatching
from agents.fictitiousplay import FictitiousPlay
from agents.random_agent import RandomAgent
import numpy as np
import itertools

## Initialize Game

Create a game with a configuration:

In [2]:
g = ThreePlayers(config=1)
print(f"Using config: {g.config}")

Using config: 1


## Reward Matrix

View the current reward matrix:

In [3]:
g.reset()

In [4]:
# Generate all possible joint actions
action_ranges = [range(g.action_spaces[agent].n) for agent in g.agents]
all_joint_actions = list(itertools.product(*action_ranges))

for joint_action in all_joint_actions:
    actions = dict(zip(g.agents, joint_action))
    print(f"Actions: {tuple(map(lambda agent: g._moves[agent][actions[agent]], g.agents))}")
    g.step(actions)
    print(f"Rewards: {tuple(map(lambda agent: float(g.rewards[agent]), g.agents))}")

Actions: ('T', 'L', 'W')
Rewards: (0.0, 0.0, 0.0)
Actions: ('T', 'L', 'E')
Rewards: (0.0, 0.0, 0.0)
Actions: ('T', 'R', 'W')
Rewards: (2.0, 8.0, 6.0)
Actions: ('T', 'R', 'E')
Rewards: (1.0, 2.0, 5.0)
Actions: ('B', 'L', 'W')
Rewards: (5.0, 3.0, 2.0)
Actions: ('B', 'L', 'E')
Rewards: (1.0, 6.0, 1.0)
Actions: ('B', 'R', 'W')
Rewards: (3.0, 4.0, 2.0)
Actions: ('B', 'R', 'E')
Rewards: (0.0, 0.0, 1.0)


## Learning with Fictitious Play

Train agents using Fictitious Play algorithm:

In [5]:
g.reset()
fp = dict(map(lambda agent: (agent, FictitiousPlay(game=g, agent=agent)), g.agents))

In [6]:
for i in range(10000):
    actions = dict(map(lambda agent: (agent, fp[agent].action()), g.agents))
    g.step(actions)

### Learned Policies (Fictitious Play)

In [7]:
dict(map(lambda agent: (agent, fp[agent].policy()), g.agents))

{'agent_0': array([6.99161007e-04, 9.99300839e-01]),
 'agent_1': array([5.99460486e-04, 9.99400540e-01]),
 'agent_2': array([9.99200959e-01, 7.99041151e-04])}

### Expected Rewards at Learned Equilibrium

In [8]:
g._R[tuple(map(lambda agent: np.argmax(fp[agent].policy()), g.agents))]

array([3., 4., 2.])

## Learning with Regret Matching

Train agents using Regret Matching algorithm:

In [9]:
g.reset()
rm = dict(map(lambda agent: (agent, RegretMatching(game=g, agent=agent)), g.agents))

In [10]:
for i in range(10000):
    actions = dict(map(lambda agent: (agent, rm[agent].action()), g.agents))
    g.step(actions)

### Learned Policies (Regret Matching)

In [11]:
dict(map(lambda agent: (agent, rm[agent].policy()), g.agents))

{'agent_0': array([2.5000e-04, 9.9975e-01]),
 'agent_1': array([5.0000e-05, 9.9995e-01]),
 'agent_2': array([9.999e-01, 1.000e-04])}

### Expected Rewards at Learned Equilibrium

In [12]:
g._R[tuple(map(lambda agent: np.argmax(rm[agent].policy()), g.agents))]

array([3., 4., 2.])